# 01. Data Preprocessing

Notebook ini bertugas membersihkan data mentah hasil temuan pada `00_Data_Exploration.ipynb` agar siap untuk tahap seleksi fitur & feature engineering. Tugas notebook ini **hanya sampai data bersih**, belum melakukan seleksi fitur/imputasi top-15/normalisasi/split (itu tugas `02_Feature_Engineering.ipynb`).

**Pipeline notebook:** `00_Data_Exploration` → `01_Data_Preprocessing` → `02_Feature_Engineering` → `03_Modeling_ANN` → `04_Training_Evaluation` → `05_SHAP_Explainability` → `06_Deployment`

**Tahapan pada notebook ini:**
1. Menghapus kolom tidak relevan
2. Menghapus kolom dengan missing value > 70%
3. Deteksi & hapus data duplikat
4. Konversi tipe data (`term`, `emp_length`)
5. Menghapus kolom kategorikal berkardinalitas tinggi
6. Encoding label target (loan_status → integer)
7. **Menyimpan hasil (`data/processed/loan_cleaned.parquet` + mapping target)** agar bisa dipakai notebook selanjutnya tanpa mengulang dari raw data.

---

## Import Library

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import warnings
import json
import sys, os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Load Data

In [2]:
path = "../data/raw"
df = pd.read_csv(os.path.join(path, 'loan.csv'), low_memory=False)

print(f"Dataset shape: {df.shape[0]:,} baris × {df.shape[1]} kolom")
df.head(3)

Dataset shape: 887,379 baris × 74 kolom


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_il_6m,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,1077501,1296599,5000.00,5000.00,4975.00,36 months,10.65,162.87,B,B2,NaN,10+ years,RENT,24000.00,Verified,Dec-2011,Fully Paid,n,https://www.lendingclub.com/browse/loanDetail....,Borrower added on 12/22/11 > I need to upgra...,credit_card,Computer,860xx,AZ,27.65,0.00,Jan-1985,1.00,NaN,NaN,3.00,0.00,13648.00,83.70,9.00,f,0.00,0.00,5861.07,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-2015,171.62,NaN,Jan-2016,0.00,NaN,1.00,INDIVIDUAL,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1077430,1314167,2500.00,2500.00,2500.00,60 months,15.27,59.83,C,C4,Ryder,< 1 year,RENT,30000.00,Source Verified,Dec-2011,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,Borrower added on 12/22/11 > I plan to use t...,car,bike,309xx,GA,1.00,0.00,Apr-1999,5.00,NaN,NaN,3.00,0.00,1687.00,9.40,4.00,f,0.00,0.00,1008.71,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-2013,119.66,NaN,Sep-2013,0.00,NaN,1.00,INDIVIDUAL,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1077175,1313524,2400.00,2400.00,2400.00,36 months,15.96,84.33,C,C5,NaN,10+ years,RENT,12252.00,Not Verified,Dec-2011,Fully Paid,n,https://www.lendingclub.com/browse/loanDetail....,NaN,small_business,real estate business,606xx,IL,8.72,0.00,Nov-2001,2.00,NaN,NaN,2.00,0.00,2956.00,98.50,10.00,f,0.00,0.00,3003.65,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-2014,649.91,NaN,Jan-2016,0.00,NaN,1.00,INDIVIDUAL,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Menghapus kolom tidak relevan

In [3]:
# Identifier
drop_cols = ['id', 'member_id', 'url']

# Teks bebas
drop_cols += ['desc', 'emp_title', 'title']

# Konstan
drop_cols += ['policy_code']

# Data leakage (kolom yg cuma ada SETELAH pinjaman cair/berjalan)
leakage_kw = ['pymnt', 'payment', 'recover', 'out_prncp',
              'total_pymnt', 'total_rec', 'settlement', 'hardship', 'chargeoff']
drop_cols += [c for c in df.columns if any(k in c.lower() for k in leakage_kw)]
drop_cols += ['collection_recovery_fee']  # leakage manual, lihat catatan sebelumnya

# --- Fitur baru: umur riwayat kredit (bukan leakage, diketahui SAAT pengajuan) ---
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
df['credit_history_months'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 30.44
).round(1)

# Kolom tanggal mentah dibuang SETELAH dipakai di atas
# (last_credit_pull_d/last_pymnt_d/next_pymnt_d = leakage,
#  issue_d/earliest_cr_line = sudah diekstrak jadi credit_history_months)
drop_cols += ['issue_d', 'last_credit_pull_d', 'earliest_cr_line',
              'last_pymnt_d', 'next_pymnt_d']

# Redundan
drop_cols += ['funded_amnt', 'funded_amnt_inv']

drop_cols = [c for c in set(drop_cols) if c in df.columns]
df = df.drop(columns=drop_cols).copy()

print(f"Kolom dihapus : {len(drop_cols)}")
print(f"Shape sekarang: {df.shape}")
print(f"\nDistribusi credit_history_months:\n{df['credit_history_months'].describe()}")

Kolom dihapus : 25
Shape sekarang: (887379, 50)

Distribusi credit_history_months:
count   887350.00
mean       196.11
std         89.56
min          6.00
25%        135.00
50%        178.00
75%        241.00
max        851.90
Name: credit_history_months, dtype: float64


## Drop kolom dengan missing values > 70%

In [4]:
high_missing = df.isnull().mean()
high_missing = high_missing[high_missing > 0.70].index.tolist()

print(f"Kolom dihapus (missing > 70%): {len(high_missing)}")
print(high_missing)

df = df.drop(columns=high_missing)
print(f"\nShape sekarang: {df.shape}")

Kolom dihapus (missing > 70%): 19
['mths_since_last_record', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m']

Shape sekarang: (887379, 31)


## Deteksi dan hapus duplikat

In [5]:
duplicate_count = df.duplicated().sum()
print(f"Jumlah data duplikat ditemukan: {duplicate_count}")

if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print("Data duplikat telah dihapus.")
else:
    print("Tidak ada data duplikat.")

print(f"Ukuran dataset sekarang: {df.shape}")

Jumlah data duplikat ditemukan: 0
Tidak ada data duplikat.
Ukuran dataset sekarang: (887379, 31)


## Konversi tipe data

In [6]:
import re
# term
if 'term' in df.columns:
    df['term'] = df['term'].astype(str).str.extract(r'(\d+)').astype(float)

# emp_length
def parse_emp_length(s):
    if pd.isna(s):
        return np.nan
    s = str(s)
    if '<' in s:
        return 0.5
    if '10+' in s:
        return 10.0
    m = re.search(r'(\d+)', s)
    return float(m.group(1)) if m else np.nan

if 'emp_length' in df.columns:
    df['emp_length'] = df['emp_length'].apply(parse_emp_length)

print("Konversi selesai.")
check_cols = [c for c in ['int_rate', 'revol_util', 'term', 'emp_length'] if c in df.columns]
display(df[check_cols].describe())

Konversi selesai.


,int_rate,revol_util,term,emp_length
count,887379.00,886877.00,887379.00,842554.00
mean,13.25,55.07,43.20,6.05
std,4.38,23.83,11.00,3.60
min,5.32,0.00,36.00,0.50
25%,9.99,37.70,36.00,3.00
50%,12.99,56.00,36.00,6.00
75%,16.20,73.60,60.00,10.00
max,28.99,892.30,60.00,10.00


## Drop kolom kategorikal dengan kardinalitas tinggi

In [7]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'loan_status']

cardinality = df[cat_cols].nunique().sort_values(ascending=False)
print("Kardinalitas kolom kategorikal:")
display(cardinality.to_frame('Unique Values'))

high_card = cardinality[cardinality > 50].index.tolist()
print(f"\nKolom dihapus (> 50 unique): {high_card}")

df = df.drop(columns=high_card, errors='ignore')
print(f"Shape sekarang: {df.shape}")

Kardinalitas kolom kategorikal:


,Unique Values
zip_code,935
addr_state,51
sub_grade,35
purpose,14
grade,7
home_ownership,6
verification_status,3
initial_list_status,2
application_type,2



Kolom dihapus (> 50 unique): ['zip_code', 'addr_state']
Shape sekarang: (887379, 29)


## Encoding Target

In [8]:
# Buat mapping berdasarkan frekuensi kelas (kelas terbanyak = 0)
status_mapping = {status: idx for idx, status in
                  enumerate(df['loan_status'].value_counts().index)}
inverse_mapping = {v: k for k, v in status_mapping.items()}

df['target'] = df['loan_status'].map(status_mapping)
df = df.drop(columns=['loan_status'])

print("Mapping target (label → angka):")
for k, v in status_mapping.items():
    print(f"  {v:>2}  →  {k}")

print(f"\nDistribusi target:\n{df['target'].value_counts().sort_index().to_string()}")

Mapping target (label → angka):
   0  →  Current
   1  →  Fully Paid
   2  →  Charged Off
   3  →  Late (31-120 days)
   4  →  Issued
   5  →  In Grace Period
   6  →  Late (16-30 days)
   7  →  Does not meet the credit policy. Status:Fully Paid
   8  →  Default
   9  →  Does not meet the credit policy. Status:Charged Off

Distribusi target:
target
0    601779
1    207723
2     45248
3     11591
4      8460
5      6253
6      2357
7      1988
8      1219
9       761


## Menyimpan Hasil Preprocessing

In [9]:
def save_parquet(df: pd.DataFrame, filename, folder="../data/processed"):
    path = os.path.join(folder, filename)
    df.to_parquet(path, index=False)
    return path

def save_json(obj, filename, folder="../artifacts"):
    path = os.path.join(folder, filename)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    return path

In [10]:
cleaned_path = save_parquet(df, 'loan_cleaned_v2.parquet')
mapping_path = save_json(status_mapping, 'status_mapping_v2.json')
inverse_mapping_path = save_json(
    {str(k): v for k, v in inverse_mapping.items()}, 'inverse_mapping_v2.json'
)

print(f'Data bersih disimpan di: {cleaned_path}')
print(f'Shape akhir: {df.shape}')
print(f'Mapping target disimpan di: {mapping_path}')

Data bersih disimpan di: ../data/processed\loan_cleaned_v2.parquet
Shape akhir: (887379, 29)
Mapping target disimpan di: ../artifacts\status_mapping_v2.json


## Ringkasan & Langkah Selanjutnya

Data telah dibersihkan dari kolom tidak relevan, kolom bocor informasi, kolom duplikat, dan kolom kategorikal berkardinalitas tinggi, serta label target sudah dalam bentuk integer. Hasil disimpan di `data/processed/loan_cleaned.parquet`.
